<a href="https://colab.research.google.com/github/c7r7/aac-chatbot/blob/main/Project_AAC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install unsloth transformers trl faiss-cpu PyMuPDF pdfplumber
import nltk
nltk.download('punkt_tab')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.7/192.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
# =======================================
# ✅ Complete Colab Cell for Inference UI
# With fixed autocomplete (no extra speaker names)
# =======================================

import os, fitz, faiss, torch, nltk
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
import torch
import nltk
nltk.download("punkt")

# ======== Load Finetuned Model ========
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name="./finetune_model",
    max_seq_length=2048,
    load_in_4bit=True
)
inference_model.eval()
# torch._dynamo.disable()

# ======== Load Knowledge Base (RAG) ========
def smart_chunk(text, chunk_size=3):
    sentences = sent_tokenize(text)
    return [" ".join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

def load_all_pdfs(folder_path="/content/user_data"):
    all_chunks = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            doc = fitz.open(os.path.join(folder_path, file))
            text = ""
            for page in doc:
                text += page.get_text()
            for para in text.split("\n\n"):
                all_chunks.extend([c.strip() for c in smart_chunk(para) if c.strip()])
    return all_chunks

knowledge_base = load_all_pdfs()
embedder = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True, normalize_embeddings=True)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

def retrieve_chunks(query, top_k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    _, indices = index.search(q_emb, top_k)
    return [knowledge_base[i] for i in indices[0]]

def generate_response(context, query):
    prompt = f"ONLY use this context to help the AAC user reply:\n{context}\n\nPerson: {query}\nYou:"
    inputs = inference_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = inference_model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            pad_token_id=inference_tokenizer.pad_token_id
        )
    decoded = inference_tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded.split("You:")[-1].strip()

def autocomplete_suggestions(partial):
    context_chunks = retrieve_chunks(partial)
    context = "\n".join(context_chunks)
    prompt = f"{partial}"
    inputs = inference_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=15,
            temperature=0.7,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=4,
            pad_token_id=inference_tokenizer.pad_token_id
        )
    suggestions = []
    for out in outputs:
        decoded = inference_tokenizer.decode(out, skip_special_tokens=True)
        suggestion = decoded.strip()
        if suggestion and suggestion not in suggestions:
            suggestions.append(suggestion)
    return suggestions[:4]

# ========================== UI SETUP ==========================
tarun_input = widgets.Text(placeholder='Tarun says...', description='Tarun:', layout=widgets.Layout(width='700px'))
tarun_send = widgets.Button(description='Send →', button_style='primary')

custom_input = widgets.Text(placeholder='Type your reply...', description='You:', layout=widgets.Layout(width='700px'))
custom_submit = widgets.Button(description='Submit Reply', button_style='success')

chat_display = widgets.HTML(value="<b>Conversation:</b><br>")
suggestion_box = widgets.HTML()
suggestion_buttons = widgets.VBox()
autocomplete_box = widgets.HTML()
autocomplete_buttons = widgets.VBox()

chat_log = []

# === Handlers ===
def handle_tarun_send(b):
    msg = tarun_input.value.strip()
    if not msg: return
    tarun_input.value = ""
    chat_log.append(f"<b>Tarun:</b> {msg}")
    chunks = retrieve_chunks(msg)
    context = "\n".join(chunks)
    responses = [generate_response(context, msg) for _ in range(3)]
    btns = []
    for resp in responses:
        btn = widgets.Button(description=resp, layout=widgets.Layout(width='auto'))
        def on_click(b, val=resp):
            custom_input.value = val
        btn.on_click(on_click)
        btns.append(btn)
    suggestion_box.value = "<b>Suggestions:</b>"
    suggestion_buttons.children = btns
    update_chat()

def handle_custom_submit(b):
    msg = custom_input.value.strip()
    if not msg: return
    custom_input.value = ""
    chat_log.append(f"<b>You:</b> {msg}")
    suggestion_box.value = ""
    suggestion_buttons.children = []
    autocomplete_box.value = ""
    autocomplete_buttons.children = []
    update_chat()

def handle_custom_typing(change):
    text = change['new'].strip()
    if not text:
        autocomplete_box.value = ""
        autocomplete_buttons.children = []
        return
    try:
        suggestions = autocomplete_suggestions(text)
        buttons = []
        for s in suggestions:
            btn = widgets.Button(description=s, layout=widgets.Layout(width='auto'))
            def on_click(b, val=s):
                custom_input.value = val
            btn.on_click(on_click)
            buttons.append(btn)
        autocomplete_box.value = "<b>Continue with:</b>"
        autocomplete_buttons.children = buttons
    except Exception as e:
        autocomplete_box.value = f"<i>Error: {e}</i>"

def update_chat():
    clear_output(wait=True)
    chat_display.value = "<b>Conversation:</b><br>" + "<br>".join(chat_log[-20:])
    display(
        widgets.HBox([tarun_input, tarun_send]),
        chat_display,
        suggestion_box,
        suggestion_buttons,
        custom_input,
        autocomplete_box,
        autocomplete_buttons,
        custom_submit
    )

# === Event binding ===
tarun_send.on_click(handle_tarun_send)
custom_submit.on_click(handle_custom_submit)
custom_input.observe(handle_custom_typing, names="value")

# Launch
update_chat()


HTML(value='<b>Conversation:</b><br><b>Tarun:</b> whats your cgpa<br><b>You:</b> Mine is not that gooey. I gue…

HTML(value='')

VBox()

Text(value='', description='You:', layout=Layout(width='700px'), placeholder='Type your reply...')

HTML(value='')

VBox()

Button(button_style='success', description='Submit Reply', style=ButtonStyle())

In [1]:
# ============================================================
# ✅ AAC Chat UI + LangChain Retriever + Memory + Unsloth LLM
# ============================================================

!pip install -q langchain sentence-transformers faiss-cpu transformers bitsandbytes unsloth ipywidgets

import os, fitz, faiss, torch, nltk
import ipywidgets as widgets
import numpy as np
from IPython.display import display, clear_output
from nltk.tokenize import sent_tokenize
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS as LC_FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.llms.base import LLM
from pydantic import PrivateAttr

nltk.download("punkt")

# ====== Load BGE embedder & chunk docs ======
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5")

def smart_chunk(text, chunk_size=3):
    sentences = sent_tokenize(text)
    return [" ".join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

def load_all_pdfs(folder_path="/content/user_data"):
    all_chunks = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            doc = fitz.open(os.path.join(folder_path, file))
            text = "".join([page.get_text() for page in doc])
            for para in text.split("\n\n"):
                all_chunks.extend([c.strip() for c in smart_chunk(para) if c.strip()])
    return all_chunks

chunks = load_all_pdfs()
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
vectorstore = LC_FAISS.from_texts(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ====== Wrap your Unsloth LLM for LangChain ======
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name="./finetune_model",  # change if needed
    max_seq_length=2048,
    load_in_4bit=True
)
inference_model.eval()

class UnslothLLM(LLM):
    temperature: float = 0.7
    max_new_tokens: int = 60
    _model: any = PrivateAttr()
    _tokenizer: any = PrivateAttr()

    def __init__(self, model, tokenizer, **kwargs):
        super().__init__(**kwargs)
        self._model = model
        self._tokenizer = tokenizer

    @property
    def _llm_type(self):
        return "unsloth"

    def _call(self, prompt, stop=None):
        inputs = self._tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            output = self._model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=True,
                pad_token_id=self._tokenizer.pad_token_id
            )
        return self._tokenizer.decode(output[0], skip_special_tokens=True).strip()

llm = UnslothLLM(inference_model, inference_tokenizer)

# ====== LangChain Conversational Chain (RAG + Memory) ======
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
from langchain.prompts import PromptTemplate

custom_template = """ONLY use the following context to respond naturally.

Context:
{context}

Conversation History:
{chat_history}

Tarun: {question}
You:"""

prompt = PromptTemplate(
    input_variables=["context", "question", "chat_history"],
    template=custom_template
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=False,
)

# qa_chain = ConversationalRetrievalChain.from_llm(
#     llm=llm,
#     retriever=retriever,
#     memory=memory,
#     return_source_documents=False,
# )

# ====== Autocomplete function ======
def autocomplete_suggestions(partial):
    inputs = inference_tokenizer(partial, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=4,
            pad_token_id=inference_tokenizer.pad_token_id
        )
    results = []
    for out in outputs:
        suggestion = inference_tokenizer.decode(out, skip_special_tokens=True).strip()
        if suggestion and suggestion not in results:
            results.append(suggestion)
    return results[:4]

# ====== UI Setup ======
tarun_input = widgets.Text(placeholder='Tarun says...', description='Tarun:', layout=widgets.Layout(width='700px'))
tarun_send = widgets.Button(description='Send →', button_style='primary')

custom_input = widgets.Text(placeholder='Type your reply...', description='You:', layout=widgets.Layout(width='700px'))
custom_submit = widgets.Button(description='Submit Reply', button_style='success')

chat_display = widgets.HTML(value="<b>Conversation:</b><br>")
suggestion_box = widgets.HTML()
suggestion_buttons = widgets.VBox()
autocomplete_box = widgets.HTML()
autocomplete_buttons = widgets.VBox()

chat_log = []

# ====== UI Handlers ======
def update_chat():
    clear_output(wait=True)
    chat_display.value = "<b>Conversation:</b><br>" + "<br>".join(chat_log[-20:])
    display(
        widgets.HBox([tarun_input, tarun_send]),
        chat_display,
        suggestion_box,
        suggestion_buttons,
        custom_input,
        autocomplete_box,
        autocomplete_buttons,
        custom_submit
    )

def handle_tarun_send(_):
    msg = tarun_input.value.strip()
    if not msg: return
    tarun_input.value = ""
    chat_log.append(f"<b>Tarun:</b> {msg}")

    responses = [qa_chain.run(msg) for _ in range(3)]

    buttons = []
    for resp in responses:
        btn = widgets.Button(description=resp, layout=widgets.Layout(width='auto'))
        def on_click(b, val=resp):
            custom_input.value = val
        btn.on_click(on_click)
        buttons.append(btn)

    suggestion_box.value = "<b>Suggestions:</b>"
    suggestion_buttons.children = buttons
    update_chat()

def handle_custom_submit(_):
    msg = custom_input.value.strip()
    if not msg: return
    chat_log.append(f"<b>You:</b> {msg}")
    custom_input.value = ""
    suggestion_box.value = ""
    suggestion_buttons.children = []
    autocomplete_box.value = ""
    autocomplete_buttons.children = []
    update_chat()

def handle_custom_typing(change):
    text = change["new"].strip()
    if not text:
        autocomplete_box.value = ""
        autocomplete_buttons.children = []
        return
    try:
        suggestions = autocomplete_suggestions(text)
        buttons = []
        for s in suggestions:
            btn = widgets.Button(description=s, layout=widgets.Layout(width='auto'))
            def on_click(b, val=s):
                custom_input.value = val
            btn.on_click(on_click)
            buttons.append(btn)
        autocomplete_box.value = "<b>Continue with:</b>"
        autocomplete_buttons.children = buttons
    except Exception as e:
        autocomplete_box.value = f"<i>Error generating suggestions: {e}</i>"

# Bind events and launch
tarun_send.on_click(handle_tarun_send)
custom_submit.on_click(handle_custom_submit)
custom_input.observe(handle_custom_typing, names="value")
update_chat()


HTML(value='<b>Conversation:</b><br><b>Tarun:</b> what are the companies that you worked for?')

HTML(value='<b>Suggestions:</b>')

Text(value='', description='You:', layout=Layout(width='700px'), placeholder='Type your reply...')

HTML(value='')

VBox()

Button(button_style='success', description='Submit Reply', style=ButtonStyle())

In [12]:
from langchain.llms.base import LLM
from pydantic import PrivateAttr

class UnslothLLM(LLM):
    model: str = "unsloth"
    temperature: float = 0.7
    max_new_tokens: int = 60

    _model: any = PrivateAttr()
    _tokenizer: any = PrivateAttr()

    def __init__(self, model, tokenizer, **kwargs):
        super().__init__(**kwargs)
        self._model = model
        self._tokenizer = tokenizer

    @property
    def _llm_type(self):
        return "unsloth"

    def _call(self, prompt, stop=None):
        inputs = self._tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            output = self._model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=True,
                pad_token_id=self._tokenizer.pad_token_id
            )
        return self._tokenizer.decode(output[0], skip_special_tokens=True).strip()


In [9]:
pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
